# Eventbrite Scrape + Zero-Shot Classifier

This notebook:
1. Scrapes Chicago Eventbrite events (name, description, date, venue)
2. Uses a zero-shot classifier to re-tag each event with **Yelp-style category labels**
3. Saves the result for use in the recommendation pipeline

> **Note:** Scraping violates Eventbrite's ToS. This is intended for academic/research use only at small scale.

In [2]:
!pip install requests beautifulsoup4 transformers torch pandas --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re
import pandas as pd
from datetime import datetime, timezone, timedelta

# -------------------------
# CONFIG
# -------------------------
CITY          = "il--chicago"
MAX_PAGES     = 25          # 25 pages × ~20 events = ~500 events
DAYS_AHEAD    = 14
OUTPUT_CSV    = "chicago_eventbrite_scraped.csv"
SLEEP_BETWEEN = 1.5         # seconds between requests — be polite

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

DATE_START = datetime.now(timezone.utc).strftime("%Y-%m-%d")
DATE_END   = (datetime.now(timezone.utc) + timedelta(days=DAYS_AHEAD)).strftime("%Y-%m-%d")
print(f"Scraping Chicago events from {DATE_START} to {DATE_END}")

Scraping Chicago events from 2026-02-23 to 2026-03-09


## Step 1 — Collect Event URLs from Search Pages

In [2]:
def get_event_urls_from_page(page_num):
    """Scrape one search results page and return all event URLs found."""
    url = (
        f"https://www.eventbrite.com/d/{CITY}/all-events/"
        f"?start_date={DATE_START}&end_date={DATE_END}&page={page_num}"
    )
    resp = requests.get(url, headers=HEADERS, timeout=15)
    if resp.status_code != 200:
        print(f"  Page {page_num}: HTTP {resp.status_code}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")

    # Event links look like /e/event-name-tickets-XXXXXXXXX
    links = set()
    for tag in soup.find_all("a", href=True):
        href = tag["href"]
        if "/e/" in href and "tickets" in href:
            # Normalise to base URL (strip query params)
            base = href.split("?")[0]
            if base.startswith("/"):
                base = "https://www.eventbrite.com" + base
            links.add(base)

    # Also try extracting from embedded JSON (Eventbrite sometimes puts data here)
    for script in soup.find_all("script", type="application/json"):
        try:
            data = json.loads(script.string)
            text = json.dumps(data)
            found = re.findall(r'https://www\.eventbrite\.com/e/[^"\s]+', text)
            for f in found:
                links.add(f.split("?")[0])
        except Exception:
            pass

    return list(links)


all_urls = []
for p in range(1, MAX_PAGES + 1):
    urls = get_event_urls_from_page(p)
    all_urls.extend(urls)
    print(f"Page {p}: found {len(urls)} event URLs (running total: {len(set(all_urls))})")
    time.sleep(SLEEP_BETWEEN)

all_urls = list(set(all_urls))
print(f"\nTotal unique event URLs: {len(all_urls)}")

Page 1: found 18 event URLs (running total: 18)
Page 2: found 18 event URLs (running total: 36)
Page 3: found 20 event URLs (running total: 56)
Page 4: found 19 event URLs (running total: 75)
Page 5: found 19 event URLs (running total: 94)
Page 6: found 20 event URLs (running total: 114)
Page 7: found 19 event URLs (running total: 133)
Page 8: found 20 event URLs (running total: 153)
Page 9: found 20 event URLs (running total: 173)
Page 10: found 20 event URLs (running total: 193)
Page 11: found 20 event URLs (running total: 213)
Page 12: found 20 event URLs (running total: 233)
Page 13: found 20 event URLs (running total: 253)
Page 14: found 20 event URLs (running total: 273)
Page 15: found 20 event URLs (running total: 293)
Page 16: found 18 event URLs (running total: 311)
Page 17: found 20 event URLs (running total: 331)
Page 18: found 20 event URLs (running total: 351)
Page 19: found 18 event URLs (running total: 369)
Page 20: found 19 event URLs (running total: 388)
Page 21: found

In [3]:
def extract_overview_text(soup):
    """
    Extract the organizer-written overview/description body from an event page.
    Tries multiple selectors since Eventbrite's HTML structure varies.
    """
    # Selectors to try in order of reliability
    selectors = [
        {"data-testid": "event-description"},
        {"class": "structured-content-rich-text"},
        {"class": "event-description__content"},
        {"class": "eds-text--left"},
        {"class": "summary"},
    ]
    for attrs in selectors:
        tag = soup.find(["div", "section", "p"], attrs=attrs)
        if tag:
            text = tag.get_text(separator=" ", strip=True)
            if len(text) > 40:   # ignore trivially short matches
                return text

    # Broader fallback: find the longest <p> block on the page
    paragraphs = soup.find_all("p")
    if paragraphs:
        longest = max(paragraphs, key=lambda t: len(t.get_text()))
        text = longest.get_text(separator=" ", strip=True)
        if len(text) > 40:
            return text

    return ""


def scrape_event_page(url):
    """Scrape a single event page. Returns a dict or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "html.parser")

        # --- Get name from JSON-LD ---
        name       = ""
        start_date = ""
        end_date   = ""
        venue_name = ""
        venue_city = ""
        venue_state = ""

        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                if isinstance(data, list):
                    data = next((d for d in data if d.get("@type") == "Event"), {})
                if data.get("@type") == "Event":
                    name        = data.get("name", "")
                    start_date  = data.get("startDate", "")
                    end_date    = data.get("endDate", "")
                    loc         = data.get("location", {})
                    addr        = loc.get("address", {})
                    venue_name  = loc.get("name", "")
                    venue_city  = addr.get("addressLocality", "")
                    venue_state = addr.get("addressRegion", "")
                    break
            except Exception:
                pass

        # Fallback: get name from page title
        if not name:
            title_tag = soup.find("title")
            if title_tag:
                name = title_tag.text.replace("| Eventbrite", "").strip()

        if not name:
            return None

        # --- Get overview text from page body (NOT from JSON-LD description) ---
        overview = extract_overview_text(soup)

        # Combine: "Event Name. Overview text"
        full_text = f"{name}. {overview}".strip() if overview else name

        return {
            "url":         url,
            "name":        name,
            "description": overview,        # raw overview for reference
            "classifier_input": full_text,  # what gets fed to BART
            "start_date":  start_date,
            "end_date":    end_date,
            "venue_name":  venue_name,
            "venue_city":  venue_city,
            "venue_state": venue_state,
        }

    except Exception as ex:
        print(f"  Error scraping {url}: {ex}")
    return None


events = []
for i, url in enumerate(all_urls):
    result = scrape_event_page(url)
    if result and result["name"]:
        events.append(result)
    if (i + 1) % 10 == 0:
        print(f"  Scraped {i+1}/{len(all_urls)}, {len(events)} valid events so far")
    time.sleep(SLEEP_BETWEEN)

events_df = pd.DataFrame(events).drop_duplicates(subset=["url"])
print(f"\nScraped {len(events_df)} events")
events_df.head(3)

  Scraped 10/486, 10 valid events so far
  Scraped 20/486, 20 valid events so far
  Scraped 30/486, 30 valid events so far
  Scraped 40/486, 40 valid events so far
  Scraped 50/486, 50 valid events so far
  Scraped 60/486, 60 valid events so far
  Scraped 70/486, 70 valid events so far
  Scraped 80/486, 80 valid events so far
  Scraped 90/486, 90 valid events so far
  Scraped 100/486, 100 valid events so far
  Scraped 110/486, 110 valid events so far
  Scraped 120/486, 120 valid events so far
  Scraped 130/486, 130 valid events so far
  Scraped 140/486, 140 valid events so far
  Scraped 150/486, 150 valid events so far
  Scraped 160/486, 160 valid events so far
  Scraped 170/486, 170 valid events so far
  Scraped 180/486, 180 valid events so far
  Scraped 190/486, 190 valid events so far
  Scraped 200/486, 200 valid events so far
  Scraped 210/486, 210 valid events so far
  Scraped 220/486, 220 valid events so far
  Scraped 230/486, 230 valid events so far
  Scraped 240/486, 240 valid 

,url,name,description,classifier_input,start_date,end_date,venue_name,venue_city,venue_state
0,https://www.eventbrite.com/e/asu-chicago-alumn...,ASU Chicago Alumni Chapter Founder's Day Schol...,This special afternoon is dedicated to celebra...,ASU Chicago Alumni Chapter Founder's Day Schol...,,,,,
1,https://www.eventbrite.com/e/pmp-certification...,PMP Certification & Training Bootcamp in Chica...,Module 1: Introduction to Project Management \...,PMP Certification & Training Bootcamp in Chica...,,,,,
2,https://www.eventbrite.com/e/bbh-day-tickets-1...,"BBH DAY Tickets, Saturday, Feb 28 at 8 pm to S...",BBH Events is bringing Chicago a new standard ...,"BBH DAY Tickets, Saturday, Feb 28 at 8 pm to S...",,,,,


In [4]:
def clean_meta_description(desc):
    """
    Eventbrite meta descriptions follow the pattern:
    'Eventbrite - {useful info} - {date} at {location}. Find event and ticket information.'
    Extract just the middle part between the first and second ' - '.
    """
    parts = desc.split(" - ")
    if len(parts) >= 3:
        return parts[1].strip()
    elif len(parts) == 2:
        # 'Eventbrite - {useful info}'
        return parts[1].strip()
    return desc.strip()


def scrape_event_page(url):
    """Scrape a single event page. Returns a dict or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "html.parser")

        name        = ""
        start_date  = ""
        end_date    = ""
        venue_name  = ""
        venue_city  = ""
        venue_state = ""
        raw_desc    = ""

        # Get name + meta description from JSON-LD
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                if isinstance(data, list):
                    data = next((d for d in data if d.get("@type") == "Event"), {})
                if data.get("@type") == "Event":
                    name        = data.get("name", "")
                    raw_desc    = data.get("description", "")
                    start_date  = data.get("startDate", "")
                    end_date    = data.get("endDate", "")
                    loc         = data.get("location", {})
                    addr        = loc.get("address", {})
                    venue_name  = loc.get("name", "")
                    venue_city  = addr.get("addressLocality", "")
                    venue_state = addr.get("addressRegion", "")
                    break
            except Exception:
                pass

        # Fallback: get name from page title
        if not name:
            title_tag = soup.find("title")
            if title_tag:
                name = title_tag.text.replace("| Eventbrite", "").strip()

        if not name:
            return None

        # Clean the meta description — extract the useful middle part
        cleaned_desc = clean_meta_description(raw_desc) if raw_desc else ""

        # Classifier input: "Event Name. Cleaned description"
        classifier_input = f"{name}. {cleaned_desc}".strip() if cleaned_desc else name

        return {
            "url":              url,
            "name":             name,
            "description":      cleaned_desc,
            "classifier_input": classifier_input,
            "start_date":       start_date,
            "end_date":         end_date,
            "venue_name":       venue_name,
            "venue_city":       venue_city,
            "venue_state":      venue_state,
        }

    except Exception as ex:
        print(f"  Error scraping {url}: {ex}")
    return None


events = []
for i, url in enumerate(all_urls):
    result = scrape_event_page(url)
    if result and result["name"]:
        events.append(result)
    if (i + 1) % 10 == 0:
        print(f"  Scraped {i+1}/{len(all_urls)}, {len(events)} valid events so far")
    time.sleep(SLEEP_BETWEEN)

events_df = pd.DataFrame(events).drop_duplicates(subset=["url"])
print(f"\nScraped {len(events_df)} events")
events_df[["name", "description", "classifier_input"]].head(5)

  Scraped 10/486, 10 valid events so far
  Scraped 20/486, 20 valid events so far
  Scraped 30/486, 30 valid events so far
  Scraped 40/486, 40 valid events so far
  Scraped 50/486, 50 valid events so far
  Scraped 60/486, 60 valid events so far
  Scraped 70/486, 70 valid events so far
  Scraped 80/486, 80 valid events so far
  Scraped 90/486, 90 valid events so far
  Scraped 100/486, 100 valid events so far
  Scraped 110/486, 110 valid events so far
  Scraped 120/486, 120 valid events so far
  Scraped 130/486, 130 valid events so far
  Scraped 140/486, 140 valid events so far
  Scraped 150/486, 150 valid events so far
  Scraped 160/486, 160 valid events so far
  Scraped 170/486, 170 valid events so far
  Scraped 180/486, 180 valid events so far
  Scraped 190/486, 190 valid events so far
  Scraped 200/486, 200 valid events so far
  Scraped 210/486, 210 valid events so far
  Scraped 220/486, 220 valid events so far
  Scraped 230/486, 230 valid events so far
  Scraped 240/486, 240 valid 

,name,description,classifier_input
0,ASU Chicago Alumni Chapter Founder's Day Schol...,,ASU Chicago Alumni Chapter Founder's Day Schol...
1,PMP Certification & Training Bootcamp in Chica...,,PMP Certification & Training Bootcamp in Chica...
2,"BBH DAY Tickets, Saturday, Feb 28 at 8 pm to S...",,"BBH DAY Tickets, Saturday, Feb 28 at 8 pm to S..."
3,The Chicago Pancakes & Booze Art Show (Vendor/...,,The Chicago Pancakes & Booze Art Show (Vendor/...
4,"Tech Connect: Startups, Investors, Professiona...",,"Tech Connect: Startups, Investors, Professiona..."


## Step 2 — Scrape Each Event Page for Name + Description

Each Eventbrite event page embeds a `<script type="application/ld+json">` block with structured data. This is server-side rendered and reliable.

In [5]:
def scrape_event_page(url):
    """Scrape a single event page. Returns a dict or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, "html.parser")

        # --- Try JSON-LD first (most reliable) ---
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string)
                # Can be a list or a single object
                if isinstance(data, list):
                    data = next((d for d in data if d.get("@type") == "Event"), {})
                if data.get("@type") == "Event":
                    loc  = data.get("location", {})
                    addr = loc.get("address", {})
                    return {
                        "url":         url,
                        "name":        data.get("name", ""),
                        "description": data.get("description", ""),
                        "start_date":  data.get("startDate", ""),
                        "end_date":    data.get("endDate", ""),
                        "venue_name":  loc.get("name", ""),
                        "venue_city":  addr.get("addressLocality", ""),
                        "venue_state": addr.get("addressRegion", ""),
                    }
            except Exception:
                pass

        # --- Fallback: grab title + meta description ---
        name = ""
        desc = ""
        title_tag = soup.find("title")
        if title_tag:
            name = title_tag.text.replace(" | Eventbrite", "").strip()
        meta = soup.find("meta", attrs={"name": "description"})
        if meta:
            desc = meta.get("content", "")
        if name:
            return {"url": url, "name": name, "description": desc,
                    "start_date": "", "end_date": "",
                    "venue_name": "", "venue_city": "", "venue_state": ""}
    except Exception as ex:
        print(f"  Error scraping {url}: {ex}")
    return None


events = []
for i, url in enumerate(all_urls):
    result = scrape_event_page(url)
    if result and result["name"]:
        events.append(result)
    if (i + 1) % 10 == 0:
        print(f"  Scraped {i+1}/{len(all_urls)} pages, {len(events)} valid events so far")
    time.sleep(SLEEP_BETWEEN)

events_df = pd.DataFrame(events).drop_duplicates(subset=["url"])
print(f"\nScraped {len(events_df)} events")
events_df.head(3)

  Scraped 10/486 pages, 10 valid events so far
  Scraped 20/486 pages, 20 valid events so far
  Scraped 30/486 pages, 30 valid events so far
  Scraped 40/486 pages, 40 valid events so far
  Scraped 50/486 pages, 50 valid events so far
  Scraped 60/486 pages, 60 valid events so far
  Scraped 70/486 pages, 70 valid events so far
  Scraped 80/486 pages, 80 valid events so far
  Scraped 90/486 pages, 90 valid events so far
  Scraped 100/486 pages, 100 valid events so far
  Scraped 110/486 pages, 110 valid events so far
  Scraped 120/486 pages, 120 valid events so far
  Scraped 130/486 pages, 130 valid events so far
  Scraped 140/486 pages, 140 valid events so far
  Scraped 150/486 pages, 150 valid events so far
  Scraped 160/486 pages, 160 valid events so far
  Scraped 170/486 pages, 170 valid events so far
  Scraped 180/486 pages, 180 valid events so far
  Scraped 190/486 pages, 190 valid events so far
  Scraped 200/486 pages, 200 valid events so far
  Scraped 210/486 pages, 210 valid eve

,url,name,description,start_date,end_date,venue_name,venue_city,venue_state
0,https://www.eventbrite.com/e/asu-chicago-alumn...,ASU Chicago Alumni Chapter Founder's Day Schol...,Eventbrite - Alabama State University Chicago ...,,,,,
1,https://www.eventbrite.com/e/pmp-certification...,PMP Certification & Training Bootcamp in Chica...,Eventbrite - iCertGlobal presents PMP Certific...,,,,,
2,https://www.eventbrite.com/e/bbh-day-tickets-1...,"BBH DAY Tickets, Saturday, Feb 28 at 8 pm to S...",Eventbrite - Julius Charles presents BBH DAY -...,,,,,


In [ ]:
events_df.iloc[0]["url"]

In [ ]:
events_df.iloc[0]["description"]

## Step 3 — Load Yelp Category Labels

We use a curated subset of the most event-relevant Yelp categories as the zero-shot label set. 600 unique Yelp categories exist but most (auto repair, tires, etc.) don't correspond to any event type. We keep ~35 that do.

In [6]:
# Curated event-relevant Yelp categories
YELP_EVENT_LABELS = [
    # Food & Drink
    "Restaurants", "Bars", "Nightlife", "Coffee & Tea",
    "Beer", "Wine & Spirits", "Sports Bars", "Breweries",
    "Specialty Food", "Barbeque",
    # Arts & Entertainment
    "Arts & Entertainment", "Music Venues", "Jazz & Blues",
    "Comedy Clubs", "Performing Arts", "Theaters",
    "Art Galleries", "Museums",
    # Active Life & Sports
    "Active Life", "Fitness & Instruction", "Yoga",
    "Bowling", "Golf", "Sports Clubs",
    # Family & Kids
    "Kids Activities", "Arcades", "Amusement Parks", "Aquariums",
    # Shopping & Lifestyle
    "Shopping", "Fashion", "Beauty & Spas",
    # Community & Events
    "Venues & Event Spaces", "Event Planning & Services",
    "Hotels & Travel", "Education",
]

print(f"Using {len(YELP_EVENT_LABELS)} candidate labels:")
print(YELP_EVENT_LABELS)

Using 35 candidate labels:
['Restaurants', 'Bars', 'Nightlife', 'Coffee & Tea', 'Beer', 'Wine & Spirits', 'Sports Bars', 'Breweries', 'Specialty Food', 'Barbeque', 'Arts & Entertainment', 'Music Venues', 'Jazz & Blues', 'Comedy Clubs', 'Performing Arts', 'Theaters', 'Art Galleries', 'Museums', 'Active Life', 'Fitness & Instruction', 'Yoga', 'Bowling', 'Golf', 'Sports Clubs', 'Kids Activities', 'Arcades', 'Amusement Parks', 'Aquariums', 'Shopping', 'Fashion', 'Beauty & Spas', 'Venues & Event Spaces', 'Event Planning & Services', 'Hotels & Travel', 'Education']


from transformers import pipeline
import torch

# cross-encoder/nli-deberta-v3-small:
#   - 90MB — fits easily in 4GB VRAM
#   - Purpose-built for NLI (what zero-shot uses under the hood)
#   - Only slightly less accurate than bart-large-mnli for this task
#
# Change device=0 to device=-1 to force CPU if you hit memory errors

device = 0 if torch.cuda.is_available() else -1
print(f"Using {'GPU' if device == 0 else 'CPU'}")

print("Loading zero-shot classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)
print("Model loaded.")

In [7]:
from transformers import pipeline

print("Loading zero-shot classifier (first run downloads ~1.6 GB)...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1   # use CPU; change to 0 if you have a GPU
)
print("Model loaded.")

Loading zero-shot classifier (first run downloads ~1.6 GB)...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Model loaded.


In [8]:
def build_input_text(row):
    """Combine name + description into classifier input. Cap length to avoid slow inference."""
    name = str(row.get("name", "")).strip()
    desc = str(row.get("description", "")).strip()
    text = f"{name}. {desc}" if desc else name
    return text[:512]   # BART handles up to 1024 tokens; 512 chars is plenty


def classify_event(row, top_n=3, threshold=0.30):
    """Return top_n Yelp labels whose score exceeds threshold."""
    text   = build_input_text(row)
    result = classifier(text, YELP_EVENT_LABELS, multi_label=True)
    labels = [
        label for label, score
        in zip(result["labels"], result["scores"])
        if score >= threshold
    ][:top_n]
    scores = [
        round(score, 3) for score
        in result["scores"]
        if score >= threshold
    ][:top_n]
    return labels, scores


# Run classification
print(f"Classifying {len(events_df)} events...")
yelp_labels_list  = []
yelp_scores_list  = []

for i, (_, row) in enumerate(events_df.iterrows()):
    labels, scores = classify_event(row)
    yelp_labels_list.append(labels)
    yelp_scores_list.append(scores)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(events_df)} classified")

events_df["yelp_labels"] = yelp_labels_list
events_df["yelp_scores"] = yelp_scores_list
print("Done.")

Classifying 486 events...
  10/486 classified
  20/486 classified
  30/486 classified
  40/486 classified
  50/486 classified
  60/486 classified
  70/486 classified
  80/486 classified
  90/486 classified
  100/486 classified
  110/486 classified
  120/486 classified
  130/486 classified
  140/486 classified
  150/486 classified
  160/486 classified
  170/486 classified
  180/486 classified
  190/486 classified
  200/486 classified
  210/486 classified
  220/486 classified
  230/486 classified
  240/486 classified
  250/486 classified
  260/486 classified
  270/486 classified
  280/486 classified
  290/486 classified
  300/486 classified
  310/486 classified
  320/486 classified
  330/486 classified
  340/486 classified
  350/486 classified
  360/486 classified
  370/486 classified
  380/486 classified
  390/486 classified
  400/486 classified
  410/486 classified
  420/486 classified
  430/486 classified
  440/486 classified
  450/486 classified
  460/486 classified
  470/486 classif

In [9]:
# Preview results
print("--- Sample classified events ---\n")
for _, r in events_df.head(8).iterrows():
    print(f"Event:  {r['name']}")
    print(f"Labels: {r['yelp_labels']}")
    print(f"Scores: {r['yelp_scores']}")
    print()

--- Sample classified events ---

Event:  ASU Chicago Alumni Chapter Founder's Day Scholarship Fundraiser Tickets, Saturday, Mar 7 from 1 pm to 4 pm
Labels: ['Education', 'Arts & Entertainment', 'Active Life']
Scores: [0.872, 0.865, 0.863]

Event:  PMP Certification & Training Bootcamp in Chicago, IL Tickets, Multiple dates
Labels: ['Education', 'Active Life', 'Event Planning & Services']
Scores: [0.833, 0.452, 0.443]

Event:  BBH DAY Tickets, Saturday, Feb 28 at 8 pm to Sunday, Mar 1 at 1 am
Labels: ['Nightlife', 'Arts & Entertainment', 'Barbeque']
Scores: [0.971, 0.967, 0.933]

Event:  The Chicago Pancakes & Booze Art Show (Vendor/Artist Reservations) Tickets, Saturday, Feb 28 at 8 pm
Labels: ['Arts & Entertainment', 'Nightlife', 'Venues & Event Spaces']
Scores: [0.994, 0.928, 0.901]

Event:  Tech Connect: Startups, Investors, Professionals Networking Event Chicago Tickets, Monday, Feb 23 from 7 pm to 10 pm
Labels: ['Event Planning & Services', 'Venues & Event Spaces', 'Active Life']

## Step 5 — Build event_text for Embedding

Now that each event has Yelp-style labels, we build an `event_text` field using only those labels — the same format as venues. This puts both in the same semantic space.

In [10]:
def event_to_text(row):
    labels = row["yelp_labels"]
    if not labels:
        return ""
    return "Categories: " + ", ".join(labels)

events_df["event_text"] = events_df.apply(event_to_text, axis=1)

# Drop events where classifier found no labels
before = len(events_df)
events_df = events_df[events_df["event_text"].str.strip() != ""].reset_index(drop=True)
print(f"Dropped {before - len(events_df)} events with no classifier output")
print(f"Final events: {len(events_df)}")
print("\nSample event_text:")
print(events_df["event_text"].head(5).to_string())

Dropped 15 events with no classifier output
Final events: 471

Sample event_text:
0    Categories: Education, Arts & Entertainment, A...
1    Categories: Education, Active Life, Event Plan...
2    Categories: Nightlife, Arts & Entertainment, B...
3    Categories: Arts & Entertainment, Nightlife, V...
4    Categories: Event Planning & Services, Venues ...


## Step 6 — Save

In [11]:
events_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(events_df)} events to {OUTPUT_CSV}")
print(f"Columns: {list(events_df.columns)}")

Saved 471 events to chicago_eventbrite_scraped.csv
Columns: ['url', 'name', 'description', 'start_date', 'end_date', 'venue_name', 'venue_city', 'venue_state', 'yelp_labels', 'yelp_scores', 'event_text']
